In [ ]:
import sys

sys.path.append("../src")

In [ ]:
PRETRAINED_ROBERTA_PATH = "../artifacts/encoder"
DATASET_PATH = "../data/valid.csv"
DEVICE = "cuda:0"

In [ ]:
import torch

device = torch.device(DEVICE)

## Tokenizer

In [ ]:
from transformers import XLMRobertaTokenizer

tokenizer = XLMRobertaTokenizer.from_pretrained(PRETRAINED_ROBERTA_PATH)

In [ ]:
from transformers import XLMRobertaConfig, XLMRobertaModel

config = XLMRobertaConfig.from_pretrained("xlm-roberta-base")
model = XLMRobertaModel(config).to(device)

model.resize_token_embeddings(len(tokenizer))

state_dict = torch.load(f"{PRETRAINED_ROBERTA_PATH}/state_dict.pth")
model.load_state_dict(state_dict)

model.eval()

model.config.use_cache = False
model.gradient_checkpointing_enable()

In [ ]:
import pandas as pd

dataframe = pd.read_csv(DATASET_PATH)

In [ ]:
from torch.utils.data import DataLoader

from cleaner import Cleaner
from collate_fn import collate_function
from dataset import VacanciesDataset

BATCH_SIZE = 64

cleaner = Cleaner()

train_dataset = VacanciesDataset(dataframe, cleaner.process)
data_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    collate_fn=lambda data: collate_function(data, tokenizer=tokenizer),
)

In [ ]:
from pooler import Pooler

pooler = Pooler()

In [ ]:
from torch.nn import functional

total_norm_embeddings = []

with torch.no_grad():
    for batch_X, batch_y in data_loader:
        input_ids = batch_X["input_ids"].to(device)  # dimension (BATCH_SIZE, 512)
        attention_mask = batch_X["attention_mask"].to(device)  # dimension (BATCH_SIZE, 512)

        out = model(input_ids=input_ids, attention_mask=attention_mask)
        mean_embeddings = pooler(attention_mask=attention_mask, outputs=out)

        norm_embeddings = functional.normalize(mean_embeddings)
        total_norm_embeddings.append(norm_embeddings.cpu())

In [ ]:
STAFF_TYPE_MAP = {
    "class-0": 0,
    "class-1": 1,
    "class-2": 2,
}

embeddings = torch.concat(total_norm_embeddings).numpy()
labels = dataframe["staff_type"].apply(lambda x: STAFF_TYPE_MAP[x]).to_numpy()

In [ ]:
import numpy as np
from umap import UMAP

reducer = UMAP(n_components=2, n_neighbors=30, min_dist=0.05, random_state=42)

coords_2d = reducer.fit_transform(embeddings)
norms = np.linalg.norm(coords_2d, axis=1, keepdims=True)

circle_coords = coords_2d / norms

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

plt.figure(figsize=(6, 6), facecolor="#f8f9fa")

theta = np.linspace(0, 2 * np.pi, 100)
plt.plot(np.cos(theta), np.sin(theta), color="gray", linestyle="--", alpha=0.2)

cmap = ListedColormap(["#ff0008", "#3300ff", "#00f725"])

scatter = plt.scatter(
    circle_coords[:, 0],
    circle_coords[:, 1],
    c=labels,
    cmap=cmap,
    alpha=0.6,  # Полупрозрачность для эффекта плотности
    s=10,  # Размер точки
    edgecolors="none",
)

plt.xlim(-1.2, 1.2)
plt.ylim(-1.2, 1.2)
plt.gca().set_aspect("equal")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Создаем два графика рядом
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 8), facecolor="#f8f9fa")

# Настройки для обоих графиков
for ax in [ax1, ax2, ax3]:
    # Рисуем саму окружность (направляющую)
    theta = np.linspace(0, 2 * np.pi, 150)
    ax.plot(
        np.cos(theta),
        np.sin(theta),
        color="gray",
        linestyle="--",
        alpha=0.3,
        linewidth=1,
    )

    # Фиксируем оси, чтобы круг не превратился в эллипс
    ax.set_xlim(-1.1, 1.1)
    ax.set_ylim(-1.1, 1.1)
    ax.set_aspect("equal")
    ax.axis("off")

# Маски для разделения данных
mask0 = labels == 0
mask1 = labels == 1
mask2 = labels == 2

# Левый график: Класс 0 (Синий)
ax1.scatter(
    circle_coords[mask0, 0],
    circle_coords[mask0, 1],
    color="#3300ff",
    alpha=0.6,
    s=5,
    edgecolors="none",
)
ax1.set_title("class-0", fontsize=14, pad=20)

# Правый график: Класс 1 (Красный)
ax2.scatter(
    circle_coords[mask1, 0],
    circle_coords[mask1, 1],
    color="#ff0008",
    alpha=0.6,
    s=5,
    edgecolors="none",
)
ax2.set_title("class-1", fontsize=14, pad=20)


ax3.scatter(
    circle_coords[mask2, 0],
    circle_coords[mask2, 1],
    color="#00f725",
    alpha=0.6,
    s=5,
    edgecolors="none",
)
ax3.set_title("class-2", fontsize=14, pad=20)

plt.tight_layout()
plt.show()